In [ ]:
from paddleocr import PPStructureV3
import paddlex.inference.models.runners.paddle_static.config.pp_option as pp_option
pp_option.is_mkldnn_available = lambda: False  # AVX-512 workaround

pipeline = PPStructureV3(
    device="cpu",
    text_detection_model_name="PP-OCRv6_small_det",   # keep — already proven good today
    text_recognition_model_name="PP-OCRv6_small_rec",
    use_table_recognition=True,
    use_formula_recognition=True,   # ON — matches library default, real gap in our own pipeline right now
    use_doc_orientation_classify=True,  # robustness for skewed/photographed pages — free, models already load
    use_doc_unwarping=True,             # regardless (saw them in your own log output)
    text_rec_score_thresh=0.5,      # filter low-confidence OCR noise before it reaches markdown/RAG
)

results = list(pipeline.predict("/home/ravikumar/Projects/agent-framework/pdfqa-rag/data/documents/PaperText/brochure.pdf"))
for res in results:
    res.save_to_markdown("output_small_cpu")

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/UVDoc`.
Creating model: ('PP-DocBlockLayout', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/PP-DocBlockLayout`.
Creating mo

In [5]:
for res in results:
    # res.save_to_json("output_small_cpu/json")
    res.save_to_html("output_small_cpu")

In [4]:
import numpy as np
import pdfplumber
from paddlex.inference.pipelines.layout_parsing.xycut_enhanced.utils import (
    projection_by_bboxes, split_projection_profile,
)

def safe_xy_cut(boxes, indices, res, min_gap=1):
    if len(boxes) == 0:
        return
    x_sorted_indices = boxes[:, 0].argsort()
    x_sorted_boxes = boxes[x_sorted_indices]
    x_sorted_indices = np.array(indices)[x_sorted_indices]
    x_projection = projection_by_bboxes(boxes=x_sorted_boxes, axis=0)
    x_intervals = split_projection_profile(x_projection, 0, 1)
    if not x_intervals:
        return
    for x_start, x_end in zip(*x_intervals):
        mask = (x_start <= x_sorted_boxes[:, 0]) & (x_sorted_boxes[:, 0] < x_end)
        x_boxes_chunk, x_indices_chunk = x_sorted_boxes[mask], x_sorted_indices[mask]
        if len(x_boxes_chunk) == 0:
            continue
        y_sorted = x_boxes_chunk[:, 1].argsort()
        y_boxes_chunk, y_indices_chunk = x_boxes_chunk[y_sorted], x_indices_chunk[y_sorted]
        y_projection = projection_by_bboxes(boxes=y_boxes_chunk, axis=1)
        y_intervals = split_projection_profile(y_projection, 0, min_gap)
        if not y_intervals:
            continue
        if len(y_intervals[0]) == 1:
            res.extend(y_indices_chunk)
            continue
        for y_start, y_end in zip(*y_intervals):
            m = (y_start <= y_boxes_chunk[:, 1]) & (y_boxes_chunk[:, 1] < y_end)
            safe_xy_cut(y_boxes_chunk[m], y_indices_chunk[m], res, min_gap)

with pdfplumber.open("/home/ravikumar/Projects/agent-framework/pdfqa-rag/data/documents/PaperText/brochure.pdf") as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        words = page.extract_words()
        boxes = np.array([[w["x0"], w["top"], w["x1"], w["bottom"]] for w in words], dtype=int)
        order = []
        safe_xy_cut(boxes, list(range(len(words))), order)
        page_text = " ".join(words[i]["text"] for i in order)
        print(f"--- page {page_num} ---\n{page_text}\n")

--- page 1 ---
Creating PDF/UA An assertion of PDF/UA conformance places stringent demands on both document authoring software and the author responsible for the conformance status of a particular document. In PDF, text strings of arbitrary lengths may occur in almost any content stream for a wide variety of more-or- less unfortunate reasons. Developers can take very little for granted, but PDF/UA ensures a focus on what matters for accessibility purposes. PDF/UA makes it possible to deliver accessible content with the reliability that is a hallmark of PDF. To comply with PDF/UA, all content on the PDF page must be correctly categorized as artifact or real content. All real con- tent must then be correctly characterized in semantic terms and a tags tree provided, to indicate the logical reading order of the document. High quality input PDF files are required, including several specific technical requirements for font embedding and character encoding, among other requirements. Using PDF